In [ ]:
from classical_ml_ver.classical_ml_model import *
from classical_ml_ver.selection import *
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.kernel_ridge import KernelRidge
from sklearn.semi_supervised import LabelSpreading
from sklearn.manifold import SpectralEmbedding
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.metrics.pairwise import rbf_kernel



# data

In [2]:
class WineMLDataManager:
    def __init__(self, df, d_type=8, d_quality=8):
        self.df = df.copy()

        # -------- labels --------
        self.quality = df["quality"].values.astype(float)
        self.type_raw = df["type"].values
        self.type_numeric, _ = pd.factorize(self.type_raw)

        # -------- one-hot --------
        self.type_onehot = OneHotEncoder(
            sparse_output=False
        ).fit_transform(self.type_raw[:, None])

        # -------- graph --------
        G = (self.type_numeric[:, None] == self.type_numeric[None, :]).astype(float)
        G /= G.sum(axis=1, keepdims=True)

        # -------- type embedding (graph only) --------
        from sklearn.manifold import SpectralEmbedding
        Z_type = SpectralEmbedding(
            n_components=d_type,
            affinity="precomputed"
        ).fit_transform(G)

        # -------- quality embedding (graph + y) --------
        from sklearn.kernel_ridge import KernelRidge
        kr = KernelRidge(kernel="precomputed", alpha=1e-3)
        kr.fit(G, self.quality)

        zq = kr.dual_coef_[:, None]
        Z_quality = zq @ np.random.randn(1, d_quality)

        # -------- final --------
        self.x = np.concatenate([Z_type, Z_quality], axis=1)
        self.xo = np.concatenate([self.x, self.type_onehot], axis=1)

    def get_numpy(self):
        return {
            "x": self.x,
            "xo": self.xo,
            "quality": self.quality,
            "type": self.type_numeric,
        }




In [3]:
red = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
    sep=";"
)
red["type"] = "red"

white = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
    sep=";"
)
white["type"] = "white"

df_wine = pd.concat([red, white], ignore_index=True)

print(df_wine.shape)
print(df_wine.columns)
print(df_wine.head())

(6497, 13)
Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'type'],
      dtype='object')
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26     

In [4]:
dmml=WineMLDataManager(df_wine)

/opt/homebrew/lib/python3.10/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


# exp

In [5]:
X = dmml.x         # (N, D)
XO = dmml.xo       # (N, T)
Q = dmml.quality 
T = dmml.type_numeric

# ----- fitness encoder -----
fitness_encoder = FitnessEncoder(embed_dim=6)
m_target = fitness_encoder.fit_transform(XO, Q, T)

# ----- encoder (model(x)) -----
encoder = KRR_Encoder(gamma=1.0)
m_hat = encoder.fit_transform(X, m_target)

In [6]:
dist_encoder = DistEncoderGMM(
    n_components_sample=4,
    n_components_feature=2, 
)
score_target = dist_encoder.fit_transform(XO)   # (N, 2)

# ----- score inference (decoder in NN) -----
decoder = KRR_Encoder(gamma=1.0, alpha=1e-3)
score_hat = decoder.fit_transform(m_hat, score_target)

print(score_target.shape, score_hat.shape)


(6497, 2) (6497, 2)


In [7]:
reject=cov_mu_reject_np(m_hat)

In [9]:
reject

array([ True,  True,  True, ...,  True,  True,  True], shape=(6497,))

In [8]:
muo=np.ones(m_hat.shape[0])/m_hat.shape[0]
covo=np.diag(muo*(1-muo))
score_hat=decoder.transform(m_hat)
mu,cov=update_prior_gaussian_np(muo,covo,m_hat,score_hat)

llr= lrt_ni_vs_n_np(mu,cov)
chi_stat = 2 * llr

reject = (chi_stat > 3.84)

In [10]:
reject

array([ True,  True,  True, ...,  True,  True,  True], shape=(6497,))